In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")


PyTorch version: 2.6.0+cu126
CUDA Available: True
GPU Device: NVIDIA GeForce RTX 4060


In [2]:
# numpy pratice

import numpy as np

# 1. Simulate 100 samples with 5 features
X = np.random.randn(100, 5)

# 2. Standardize features (Mean=0, Std=1)
X_scaled = (X - np.mean(X, axis=0)) / np.std(X, axis=0)
print('X_scaled.shape: ', X_scaled.shape)

# 3. Add "Bias" column (a column of ones)
X_final = np.column_stack([np.ones(100), X_scaled])

print('X_final.shape: ', X_final.shape)

# 4. Predict (Dot product with random weights)
weights = np.random.randn(6, 1)
predictions = X_final @ weights

# 5. Get final class (if binary classification)
results = (predictions > 0.5).astype(int)


import numpy as np

# 1. CREATION & INSPECTION
# Create dummy features (10 samples, 3 features) and labels
X = np.random.randint(1, 100, size=(10, 3)).astype('float32')
y = np.array([0, 1, 0, 1, 0, 0, 1, 1, 0, 1])

print(f"Shape: {X.shape}, Type: {X.dtype}, Size: {X.size}")

# 2. PREPROCESSING & LOGIC (CLEANING)
# Use np.where to cap values (any value > 80 becomes 80)
X_cleaned = np.where(X > 80, 80, X)

# 3. SCALING (MATH & STATS)
# Min-Max Scaling: (x - min) / (max - min)
X_min, X_max = X_cleaned.min(axis=0), X_cleaned.max(axis=0)
X_scaled = (X_cleaned - X_min) / (X_max - X_min)

# 4. MANIPULATION (SHAPE & STRUCTURE)
# Add a bias column (ones) using column_stack
X_final = np.column_stack([np.ones(10), X_scaled])

# Ensure y is a column vector for matrix math
y_reshaped = y.reshape(-1, 1)

# 5. SELECTION & SHUFFLING
# Shuffle data using a shared index to keep X and y synced
indices = np.arange(10)
np.random.shuffle(indices)
X_train, y_train = X_final[indices], y_reshaped[indices]

# 6. LINEAR ALGEBRA & PREDICTION
# Weights for 3 features + 1 bias = 4 weights
weights = np.random.randn(4, 1)

# Matrix Multiplication (@) and Universal Function (exp) for Sigmoid
z = X_train @ weights
probs = 1 / (1 + np.exp(-z)) # Sigmoid activation

# 7. OUTPUT & EVALUATION
# Convert probabilities to class 0 or 1
preds = (probs > 0.5).astype(int)

# Check class balance and final accuracy
classes, counts = np.unique(preds, return_counts=True)
accuracy = np.mean(preds == y_train)

print(f"Classes found: {classes} with counts {counts}")
print(f"Model Accuracy: {accuracy * 100}%")


X_scaled.shape:  (100, 5)
X_final.shape:  (100, 6)
Shape: (10, 3), Type: float32, Size: 30
Classes found: [1] with counts [10]
Model Accuracy: 50.0%


In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

# Corrected URL for MovieLens 100k dataset
url = "http://files.grouplens.org/datasets/movielens/ml-100k/u.data"
# Ensure we define the names and types clearly
df = pd.read_csv(url, sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])

# Force rating to int (this solves your TypeError)
df['rating'] = df['rating'].astype(int)

# Filter for 4+ stars
positives_df = df[df['rating'] >= 4].copy()

print(f"Loaded {len(positives_df)} positive interactions.")



# Map IDs to 0-indexed integers for embeddings
user_map = {id: i for i, id in enumerate(positives_df['user_id'].unique())}
item_map = {id: i for i, id in enumerate(df['item_id'].unique())}
positives_df['user_id'] = positives_df['user_id'].map(user_map)
positives_df['item_id'] = positives_df['item_id'].map(item_map)

all_item_ids = list(item_map.values())

Loaded 55375 positive interactions.


In [4]:
class MovieLensTriplet(Dataset):
    def __init__(self, df, all_item_ids):
        self.users = df['user_id'].values
        self.pos_items = df['item_id'].values
        self.all_items = set(all_item_ids)
        # Group watched items by user for fast negative sampling
        self.user_watched = df.groupby('user_id')['item_id'].apply(set).to_dict()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        u, p = self.users[idx], self.pos_items[idx]

        # Negative Sampling: Pick an item the user hasn't interacted with
        n = p
        while n in self.user_watched[u]:
            n = np.random.choice(list(self.all_items))

        return torch.tensor(u), torch.tensor(p), torch.tensor(n)

# Initialize Loader
dataset = MovieLensTriplet(positives_df, all_item_ids)
train_loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [5]:
# ── Sample positive and negative triplets ──────────────────────────────────
# Load movie titles so samples are human-readable
titles_url = "http://files.grouplens.org/datasets/movielens/ml-100k/u.item"
titles_df = pd.read_csv(
    titles_url, sep="|", encoding="latin-1", header=None,
    usecols=[0, 1], names=["item_id", "title"]
)
# Map raw item_id → 0-indexed item_id used in the model
titles_df["item_id"] = titles_df["item_id"].map(item_map)
titles_df = titles_df.dropna(subset=["item_id"]).set_index("item_id")["title"]

# Reverse user map for display
inv_user_map = {v: k for k, v in user_map.items()}

print("=" * 70)
print(f"{'SAMPLE TRIPLETS':^70}")
print(f"{'(user_id | positive item | negative item)':^70}")
print("=" * 70)

dataset_peek = MovieLensTriplet(positives_df, all_item_ids)

for i in range(8):
    u_idx, p_idx, n_idx = dataset_peek[i]
    u_raw  = inv_user_map[u_idx.item()]
    p_title = titles_df.get(p_idx.item(), f"item_{p_idx.item()}")
    n_title = titles_df.get(n_idx.item(), f"item_{n_idx.item()}")

    print(f"\n  User {u_raw:>4}  (mapped → {u_idx.item()})")
    print(f"    [POS]  {p_title}")
    print(f"    [NEG]  {n_title}")

print("\n" + "=" * 70)
print(f"\nDataset stats:")
print(f"  Total positive interactions : {len(positives_df):,}")
print(f"  Unique users                : {positives_df['user_id'].nunique():,}")
print(f"  Unique items                : {positives_df['item_id'].nunique():,}")
print(f"  All items in catalogue      : {len(all_item_ids):,}")
print(f"  Avg positives per user      : {len(positives_df) / positives_df['user_id'].nunique():.1f}")


                           SAMPLE TRIPLETS                            
              (user_id | positive item | negative item)               

  User  298  (mapped → 0)
    [POS]  Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb (1963)
    [NEG]  Hear My Song (1991)

  User  253  (mapped → 1)
    [POS]  Jungle Book, The (1994)
    [NEG]  Leopard Son, The (1996)

  User  286  (mapped → 2)
    [POS]  Romy and Michele's High School Reunion (1997)
    [NEG]  Killing Fields, The (1984)

  User  200  (mapped → 3)
    [POS]  Star Trek: First Contact (1996)
    [NEG]  Joy Luck Club, The (1993)

  User  122  (mapped → 4)
    [POS]  Age of Innocence, The (1993)
    [NEG]  Nothing to Lose (1994)

  User  291  (mapped → 5)
    [POS]  Just Cause (1995)
    [NEG]  Hercules (1997)

  User  119  (mapped → 6)
    [POS]  Man Without a Face, The (1993)
    [NEG]  Sum of Us, The (1994)

  User  167  (mapped → 7)
    [POS]  Sabrina (1954)
    [NEG]  Inkwell, The (1994)


Dataset stats:


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTower(nn.Module):
    def __init__(self, n_users, n_items, dim=32):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        self.user_net = nn.Linear(dim, dim)
        self.item_net = nn.Linear(dim, dim)

    def forward(self, u_id, i_id):
        # Used during training — both towers run together in one pass
        return self.get_user_embs(u_id), self.get_item_embs(i_id)

    def get_user_embs(self, u_ids):
        """
        Run the user tower for a batch of user IDs.

        Steps:
          1. Embedding lookup  : u_ids (LongTensor) -> (batch, dim) float matrix.
                                 Each integer user ID is looked up in a learned
                                 weight matrix of shape (n_users, dim), returning
                                 one dense row vector per user.
          2. Linear projection : (batch, dim) -> (batch, dim).
                                 A learned linear layer rotates and scales the raw
                                 embedding into a shared dot-product space where
                                 user and item vectors can be meaningfully compared.
          3. L2 normalisation  : each row vector is divided by its own L2 norm,
                                 projecting every user onto the unit hypersphere.
                                 This makes the dot product equivalent to cosine
                                 similarity and keeps scores in [-1, 1], preventing
                                 high-norm embeddings from dominating the ranking.

        Args:
            u_ids : LongTensor of shape (batch,) — 0-indexed user IDs.

        Returns:
            FloatTensor of shape (batch, dim) — unit-norm user embeddings.

        At eval time, call this once with all user IDs to get the full
        (n_users, dim) matrix without needing a dummy item tensor.
        """
        return F.normalize(self.user_net(self.user_emb(u_ids)), p=2, dim=1)

    def get_item_embs(self, i_ids):
        """
        Run the item tower for a batch of item IDs.

        Steps:
          1. Embedding lookup  : i_ids (LongTensor) -> (batch, dim) float matrix.
                                 Each integer item ID is looked up in a separate
                                 learned weight matrix of shape (n_items, dim).
          2. Linear projection : (batch, dim) -> (batch, dim).
                                 Projects into the same shared dot-product space
                                 as the user tower — this shared space is what
                                 makes user-item similarity scores meaningful.
          3. L2 normalisation  : unit-normalises each row, same reason as the
                                 user tower — cosine similarity, bounded scores.

        Args:
            i_ids : LongTensor of shape (batch,) — 0-indexed item IDs.

        Returns:
            FloatTensor of shape (batch, dim) — unit-norm item embeddings.

        At eval time, call this once with ALL item IDs to build a catalogue
        matrix of shape (n_items, dim). Then a single matrix multiply
        user_embs @ item_embs.T gives every user's score against every item
        in one shot: (n_users, dim) @ (dim, n_items) -> (n_users, n_items).
        """
        return F.normalize(self.item_net(self.item_emb(i_ids)), p=2, dim=1)


In [7]:
# Use the TwoTower model from our previous conversation
model = TwoTower(len(user_map), len(item_map)).cuda()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.TripletMarginLoss(margin=0.2)

for epoch in range(10):
    total_loss = 0
    for u, p, n in train_loader:
        u, p, n = u.cuda(), p.cuda(), n.cuda()

        optimizer.zero_grad()
        u_emb, p_emb = model(u, p)
        _, n_emb = model(u, n)

        loss = criterion(u_emb, p_emb, n_emb)
        loss.backward() # BACKPROPAGATION LOGIC
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Avg Loss: {total_loss/len(train_loader):.4f}")




Epoch 1 | Avg Loss: 0.1775
Epoch 2 | Avg Loss: 0.1020
Epoch 3 | Avg Loss: 0.0735
Epoch 4 | Avg Loss: 0.0658
Epoch 5 | Avg Loss: 0.0624
Epoch 6 | Avg Loss: 0.0608
Epoch 7 | Avg Loss: 0.0599
Epoch 8 | Avg Loss: 0.0598
Epoch 9 | Avg Loss: 0.0597
Epoch 10 | Avg Loss: 0.0591


In [8]:
@torch.no_grad()
def evaluate_recall(model, df, k=10):
    model.eval()
    n_users = df['user_id'].nunique()
    n_items = df['item_id'].nunique()

    # Each tower runs independently — no dummy tensors needed
    item_embs = model.get_item_embs(torch.arange(n_items).cuda())
    user_embs = model.get_user_embs(torch.arange(n_users).cuda())

    # Score every user against every item: (n_users, n_items)
    scores = user_embs @ item_embs.T
    _, top_k = torch.topk(scores, k, dim=1)
    top_k = top_k.cpu().numpy()

    # Ground truth: items each user actually rated 4+
    user_positives = df.groupby('user_id')['item_id'].apply(set).to_dict()

    hits = sum(
        1 for u in range(n_users)
        if user_positives.get(u, set()) & set(top_k[u])
    )
    return hits / n_users

print(f"Final Recall@10: {evaluate_recall(model, positives_df):.2%}")


Final Recall@10: 87.69%
